In [1]:
# 1. Install Ollama (zstd is required by the installer but missing from the base Colab image)
!apt-get -qq update && apt-get -qq install -y zstd
!apt-get update && apt-get install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy

In [2]:
import os, subprocess, time

env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0"  # <--- ADD THIS LINE

subprocess.run(["bash", "-c", "fuser -k 11434/tcp || true"])
time.sleep(3)

ollama_process = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)
print("Ollama server starting (pid:", ollama_process.pid, ")")

status = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://localhost:11434/api/tags"],
    capture_output=True, text=True,
).stdout
print("Local check (expect 200):", status)

Ollama server starting (pid: 1849 )
Local check (expect 200): 200


In [3]:
# !ollama pull minicpm-v

In [4]:
# 3. Pull the models used by CanvasUI (src/lib/models.ts)
# !ollama pull qwen2.5vl:7b


In [5]:
# !ollama pull  gpt-oss:20b

In [6]:
# !ollama pull qwen2.5-coder:14b

In [7]:
# !ollama pull qwen3.6:27b

In [8]:
# !ollama pull moondream

In [9]:
# !ollama pull qwen3:8b

In [10]:
# !ollama pull deepseek-r1:14b

In [11]:
# !ollama pull llama3.1:8b

In [12]:
# !ollama pull gemma3:12b

In [13]:
# !ollama pull gemma3:4b

In [14]:
# !ollama pull qwen3:8b
# !ollama pull qwen3:14b

In [15]:
!ollama pull phi4:14b
# !ollama pull gemma3:12b


In [16]:
# !pip install -q fastapi uvicorn pyngrok

# from fastapi import FastAPI, BackgroundTasks
# from pydantic import BaseModel
# import uvicorn, uuid, threading, base64, requests, json

# app = FastAPI()
# jobs = {}

# class SubmitPayload(BaseModel):
#     image_b64: str
#     model: str = "minicpm-v"
#     prompt: str = "Extract all text from this resume image."

# def _run_inference(job_id: str, payload: SubmitPayload):
#     try:
#         response = requests.post("http://localhost:11434/api/chat", json={
#             "model": payload.model,
#             "messages": [{"role": "user", "content": payload.prompt, "images": [payload.image_b64]}],
#             "stream": True,
#         }, stream=True, timeout=600)
#         content = ""
#         for line in response.iter_lines():
#             if line:
#                 chunk = json.loads(line)
#                 content += chunk.get("message", {}).get("content", "")
#         jobs[job_id] = {"status": "done", "result": content.strip()}
#     except Exception as e:
#         jobs[job_id] = {"status": "error", "result": str(e)}

# @app.post("/submit")
# async def submit(payload: SubmitPayload, background_tasks: BackgroundTasks):
#     job_id = str(uuid.uuid4())
#     jobs[job_id] = {"status": "pending", "result": None}
#     background_tasks.add_task(_run_inference, job_id, payload)
#     return {"job_id": job_id}

# @app.get("/result/{job_id}")
# async def get_result(job_id: str):
#     return jobs.get(job_id, {"status": "not_found"})

# # Start FastAPI in background thread
# threading.Thread(
#     target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
#     daemon=True
# ).start()

# import time; time.sleep(2)  # let server start

# # Tunnel points to port 8000 (FastAPI), NOT 11434 (Ollama)
# from pyngrok import try_cloudflare
# tunnel = ngrok.connect(8000)
# print("Paste this URL in the frontend as your Ollama URL:")
# print(tunnel.public_url)


In [17]:
!pip install -q fastapi uvicorn pyngrok

from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
from typing import Optional
import uvicorn, uuid, threading, requests, json

app = FastAPI()
jobs = {}

class SubmitPayload(BaseModel):
    image_b64: Optional[str] = None                 # now optional -- text-only calls omit this
    model: str = "minicpm-v"
    prompt: str = "Extract all text from this resume image."
    system: Optional[str] = None                     # optional system prompt (rubric/scoring calls use this)
    json_schema: Optional[dict] = None                # forces structured JSON output when set
    options: Optional[dict] = None                    # passed straight through to Ollama (temperature, etc.)

def _run_inference(job_id: str, payload: SubmitPayload):
    try:
        user_message = {"role": "user", "content": payload.prompt}
        if payload.image_b64:
            user_message["images"] = [payload.image_b64]

        messages = []
        if payload.system:
            messages.append({"role": "system", "content": payload.system})
        messages.append(user_message)

        request_body = {
            "model": payload.model,
            "messages": messages,
            "stream": True,
        }
        if payload.options:
            request_body["options"] = payload.options
        if payload.json_schema:
            request_body["format"] = payload.json_schema

        response = requests.post("http://localhost:11434/api/chat", json=request_body,
                                  stream=True, timeout=600)
        content = ""
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line)
                content += chunk.get("message", {}).get("content", "")
        jobs[job_id] = {"status": "done", "result": content.strip()}
    except Exception as e:
        jobs[job_id] = {"status": "error", "result": str(e)}

@app.post("/submit")
async def submit(payload: SubmitPayload, background_tasks: BackgroundTasks):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "pending", "result": None}
    background_tasks.add_task(_run_inference, job_id, payload)
    return {"job_id": job_id}

@app.get("/result/{job_id}")
async def get_result(job_id: str):
    return jobs.get(job_id, {"status": "not_found"})

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True
).start()

import time; time.sleep(2)

from pyngrok import ngrok
# Paste your free Ngrok authtoken here (from dashboard.ngrok.com):
# ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")
tunnel = ngrok.connect(8000)
print("Paste this URL in the frontend as your Ollama URL:")
print(tunnel.public_url)


Download cloudflared...:   0%|          | 0/39261733 [00:00<?, ?it/s]

 * Running on https://suse-suggests-captured-anaheim.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics
Paste this URL in the frontend as your Ollama URL:
https://suse-suggests-captured-anaheim.trycloudflare.com


In [18]:
print("abc")

abc


In [19]:
# # 4. Expose port 11434 publicly via Cloudflare Tunnel
# !pip install -q pyngrok

# from pyngrok import ngrok

# # Cloudflare doesn't require an authentication token for quick/anonymous tunnels!
# # Start the tunnel targeting the Ollama port
# tunnel_url = ngrok.connect(11434)

# print("Your public Ollama API endpoint is:")
# print(tunnel_url.public_url)


In [20]:
!ollama list


NAME        ID              SIZE      MODIFIED       
phi4:14b    ac896e5b8b34    9.1 GB    13 seconds ago    


In [21]:
print("abc")

abc


In [22]:
!ollama list


NAME        ID              SIZE      MODIFIED       
phi4:14b    ac896e5b8b34    9.1 GB    14 seconds ago    


In [23]:
!ollama list

NAME        ID              SIZE      MODIFIED       
phi4:14b    ac896e5b8b34    9.1 GB    14 seconds ago    


In [24]:
print("abc")

abc
